# XGBoost for Classification

XGBoost for Classification follows the same **stage-wise additive boosting framework** as Gradient Boosting. Instead of training each new model on the original labels, every new regression tree learns the **pseudo-residuals (gradients)** produced by the current ensemble.

The major distinction from traditional Gradient Boosting lies in **how trees are constructed**. Instead of using conventional impurity measures (Gini, Entropy, MSE), XGBoost builds trees using mathematically derived **Similarity Scores** and **Gain**, while incorporating **L2 regularization** directly into the optimization process.

---

# 1. Architectural Overview

XGBoost constructs the final classifier by sequentially adding weak learners.

$$
F(x)=F_0(x)+F_1(x)+F_2(x)+\cdots+F_M(x)
$$

where

- $F_0(x)$ = Initial prediction
- $F_1(x),F_2(x),\ldots$ = Sequential regression trees
- $M$ = Number of boosting iterations

Each newly added tree attempts to reduce the errors made by the previous ensemble.

---

# 2. Prerequisites

To understand XGBoost Classification, familiarity with the following concepts is recommended:

1. XGBoost for Regression
2. Gradient Boosting for Classification
3. Log-Odds
4. Logistic (Sigmoid) Function
5. Pseudo-Residuals

---

# 3. Gradient Boosting vs XGBoost

| Aspect | Gradient Boosting | XGBoost |
|---------|-------------------|----------|
| Tree Construction | Standard Decision Trees | Custom Optimized Trees |
| Split Metric | Gini / Entropy / MSE | Similarity Score & Gain |
| Regularization | Mainly Learning Rate & Depth | Built-in $L_2$ Regularization ($\lambda$) |
| Missing Values | External preprocessing required | Native handling |
| Training Speed | Sequential | Highly optimized |
| Scalability | Moderate | Excellent |

---

# 4. Dataset Setup

The lecture considers a binary classification dataset.

- **Feature ($X$):** CGPA
- **Target ($y$):** Placement

$$
y=
\begin{cases}
1,&\text{Placed}\\
0,&\text{Not Placed}
\end{cases}
$$

---

# 5. Stage 1 — Initial Prediction

Unlike regression, which initializes with the mean target value, XGBoost Classification begins with the **Log-Odds**.

## Initial Log-Odds

$$
\boxed{
\text{Log-Odds}
=
\ln\left(\frac{p}{1-p}\right)
}
$$

where

- $p$ = Probability of the positive class

---

### Example

Suppose

- Positive samples = 3
- Negative samples = 2

Then

$$
p=\frac35=0.6
$$

Therefore

$$
\text{Log-Odds}
=
\ln\left(\frac{0.6}{0.4}\right)
\approx0.405
$$

This value becomes the initial prediction for **every training sample**.

---

# 6. Convert Log-Odds into Probability

The baseline Log-Odds are transformed into probabilities using the Logistic Sigmoid Function.

$$
\boxed{
P=
\frac{e^{\text{Log-Odds}}}
{1+e^{\text{Log-Odds}}}
}
$$

Substituting

$$
\text{Log-Odds}=0.405
$$

gives

$$
P=0.60
$$

Thus every observation initially receives

$$
\boxed{P=0.60}
$$

---

# 7. Compute Pseudo-Residuals

The first residuals are

$$
\boxed{
R_1
=
y-P_0
}
$$

where

- $y$ = Actual label
- $P_0$ = Initial predicted probability

These residuals become the targets for the first regression tree.

---

# 8. Build the First Regression Tree

Although this is a classification problem, XGBoost trains a **Regression Tree** because residuals are continuous.

The regression tree learns

- Input Features
- Residuals

instead of the original class labels.

---

# 9. Similarity Score

Unlike standard decision trees, XGBoost evaluates every node using the **Similarity Score**.

$$
\boxed{
SS
=
\frac{\left(\sum R_i\right)^2}
{\sum\left(P_{\text{prev},i}(1-P_{\text{prev},i})\right)+\lambda}
}
$$

where

- $R_i$ = Residuals
- $P_{\text{prev},i}$ = Previous probability estimate
- $\lambda$ = L2 regularization parameter

---

# 10. Candidate Split Points

The feature values are first sorted.

Potential split points are created by calculating the midpoint between adjacent values.

Example thresholds:

- 5.97
- 6.67
- 7.62
- 8.87

Every threshold is evaluated.

---

# 11. Gain Formula

Each candidate split is scored using

$$
\boxed{
\text{Gain}
=
SS_{\text{Left}}
+
SS_{\text{Right}}
-
SS_{\text{Parent}}
}
$$

The split with the **largest Gain** becomes the selected node.

---

### Example

| Candidate Split | Gain |
|-----------------|------|
| CGPA < 5.97 | 1.87 |
| CGPA < 6.67 | 1.13 |
| CGPA < 7.62 | 2.22 |

Therefore

$$
\boxed{
\text{Best Split}
=
\text{CGPA}<7.62
}
$$

---

# 12. Leaf Output Value

After constructing the tree, every terminal node receives a prediction value.

Unlike Gradient Boosting, XGBoost computes this value using

$$
\boxed{
\text{Leaf Output}
=
\frac{\sum R_i}
{\sum\left(P_{\text{prev},i}(1-P_{\text{prev},i})\right)+\lambda}
}
$$

---

### Example

Left Leaf

$$
\text{Output}\approx-1.11
$$

Right Leaf

$$
\text{Output}\approx1.66
$$

---

# 13. Update Ensemble Prediction

The tree output is scaled using the Learning Rate.

$$
\boxed{
F_1(x)
=
F_0(x)
+
\eta\cdot\text{Leaf Output}
}
$$

where

- $\eta$ = Learning Rate

Typical value

$$
\eta=0.3
$$

---

# 14. Convert Updated Log-Odds into Probability

The updated prediction is converted back into probability.

$$
P=
\frac{e^{F_1(x)}}
{1+e^{F_1(x)}}
$$

These updated probabilities produce smaller residuals than before.

---

# 15. Repeat Boosting

The algorithm repeats the following sequence:

1. Compute updated probabilities.
2. Compute new residuals.
3. Train another regression tree.
4. Calculate Similarity Scores.
5. Compute Gain.
6. Determine Leaf Outputs.
7. Update Log-Odds.

The process continues until the desired number of trees has been built.

---

# 16. Complete Training Algorithm

1. Compute initial Log-Odds.
2. Convert Log-Odds into probabilities.
3. Compute pseudo-residuals.
4. Train a regression tree.
5. Compute Similarity Scores.
6. Select the split with maximum Gain.
7. Compute Leaf Output Values.
8. Update Log-Odds.
9. Convert updated Log-Odds into probabilities.
10. Repeat for all boosting rounds.

---

# 17. How XGBoost Differs from Gradient Boosting

## A. Tree Construction

### Gradient Boosting

Uses conventional decision trees.

Split criteria include:

- Gini Impurity
- Entropy
- Mean Squared Error

### XGBoost

Uses entirely new optimization metrics.

- Similarity Score
- Gain

---

## B. Leaf Node Prediction

### Gradient Boosting

Uses averages or gradient-based estimates.

### XGBoost

Uses

$$
\frac{\sum R_i}
{\sum\left(P(1-P)\right)+\lambda}
$$

which naturally incorporates regularization.

---

## C. Built-in Regularization

Traditional Gradient Boosting mainly controls complexity using

- Learning Rate
- Maximum Tree Depth

XGBoost additionally introduces explicit

$$
L_2\text{ Regularization}
$$

through

$$
\lambda
$$

This suppresses overly large leaf weights and improves generalization.

---

## D. Tree Building Algorithms

### Exact Greedy Algorithm

- Evaluates every possible split.
- Highest accuracy.
- Computationally expensive.

---

### Approximate Algorithm

- Uses Quantile Sketches.
- Divides continuous values into percentile bins.
- Tests only representative split points.
- Much faster for large datasets.

---

# 18. System-Level Optimizations

Beyond algorithmic improvements, XGBoost introduces several engineering optimizations.

### Native Missing Value Handling

Automatically learns the optimal direction for missing values during tree construction.

---

### Parallel Processing

Although trees are built sequentially, feature split evaluation inside each tree is parallelized.

---

### Cache-Aware Computing

Optimizes CPU cache usage to reduce memory access latency.

---

### Out-of-Core Computation

Allows datasets larger than RAM by streaming compressed data directly from disk.

---

# 19. Summary Table

| Concept | Summary |
|----------|---------|
| Initial Model | Log-Odds |
| Initial Probability | Sigmoid(Log-Odds) |
| Weak Learner | Regression Tree |
| Residual | $y-P$ |
| Similarity Score | Uses residuals, probabilities, and $\lambda$ |
| Gain | Chooses the optimal split |
| Leaf Output | Regularized residual update |
| Learning Rate ($\eta$) | Shrinks tree contribution |
| Regularization ($\lambda$) | Controls overfitting |
| Exact Greedy | Evaluates every split |
| Approximate Algorithm | Quantile-based split search |
| Missing Values | Native support |
| Parallel Processing | Faster split computation |
| Cache Optimization | Faster memory access |
| Out-of-Core Computing | Supports datasets larger than RAM |